# 2 · Reconciliation on FABRIC — Cascade makes the keys match over the real WAN

The FABRIC counterpart to `11_reconciliation`. Runs the real two-process BB84 across the
slice with and without `--reconcile`, so you can see Alice's and Bob's keys go from
*almost*-matching (raw) to **identical** after Cascade — with the parity exchange riding
the real classical link. Prereq: notebook fabric/01 (slice + switch).

## 1 · Configuration

In [ ]:
SLICE_NAME = 'qfabric-bb84-2'                       # same slice as notebooks fabric/01 / sequence/01
SCENARIO   = 'validation/scenarios/fabric_1km.yml'
BMV2_IMAGE = 'ghcr.io/kthare10/qfabric-bmv2:latest'

# Channel mode — matches notebook sequence/01. Use 'raw'+'switch' (the P4 path) if your slice
# already runs it; 'tcp' is a simpler switch-free option (descriptors over the link).
TRANSPORT = 'raw'      # 'raw' (0x7101 through BMv2) | 'tcp' (no switch)
LOSS      = 'switch'   # 'switch' | 'model' | 'none' | 'auto'
NUM_PULSES      = 20000
SAMPLE_FRACTION = 0.2

## 2 · Load the slice

In [ ]:
import os, sys, json
from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df
from qne.config import ScenarioConfig

cfg = ScenarioConfig.from_yaml(PROJECT_DIR / SCENARIO)
fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

## 3 · Ship code + build runtime + arm the switch

In [ ]:
df.upload_project(slice_obj)
if BMV2_IMAGE:
    os.environ['QFABRIC_BMV2_IMAGE'] = BMV2_IMAGE
df.setup_sequence_runtime(slice_obj)

USE_SWITCH = (TRANSPORT == 'raw' and LOSS in ('switch', 'auto'))
if USE_SWITCH:
    df.configure_switch(slice_obj, cfg.loss_threshold_u32)
    print('switch armed')
else:
    print(f'no switch needed (transport={TRANSPORT}, loss={LOSS})')

## 4 · With vs without reconciliation (on the slice)

In [ ]:
FIDELITY = 0.95   # ~2.5% QBER — enough errors to make reconciliation visible

def run(reconcile):
    a, b = df.run_sequence_bb84(
        slice_obj, transport=TRANSPORT, loss=LOSS, num_pulses=NUM_PULSES,
        fidelity=FIDELITY, efficiency=cfg.detector.efficiency,
        dark_count_rate=cfg.detector.dark_count_rate, distance_km=cfg.channel.distance_km,
        attenuation=cfg.channel.attenuation_db_per_km, sample_fraction=SAMPLE_FRACTION,
        reconcile=reconcile)
    return a, b

raw_a, raw_b = run(False)
rec_a, rec_b = run(True)

def mism(a, b):
    return bin((a['key'] or 0) ^ (b['key'] or 0)).count('1') / max(a['key_bits'], 1) if a['key'] and b['key'] else None

print(f"\n{'':20}{'QBER':>8}{'keys match':>12}{'mismatch':>10}{'corrections':>13}{'leaked':>9}{'secure_bits':>13}")
for label, a, b in [('no reconciliation', raw_a, raw_b), ('with Cascade', rec_a, rec_b)]:
    if a and b:
        print(f"{label:20}{a['qber']:>8.4f}{str(a['key']==b['key']):>12}"
              f"{(mism(a,b) or 0):>10.4f}{str(a['corrections']):>13}"
              f"{str(a['bits_leaked']):>9}{str(a['secure_key_bits']):>13}")

## 5 · Verify

In [ ]:
checks = []
def chk(n, ok, d=''):
    checks.append(ok); print(f"  [{'PASS' if ok else 'FAIL'}] {n}" + (f' — {d}' if d else ''))
chk('noisy channel had errors', raw_a['qber'] > 0, f"QBER={raw_a['qber']:.4f}")
chk('without reconciliation keys DIFFER', raw_a['key'] != raw_b['key'])
chk('with Cascade keys MATCH bit-for-bit', rec_a['key'] == rec_b['key'])
chk('Cascade did work', rec_a['corrections'] > 0 and rec_a['bits_leaked'] > 0)
chk('reconciled run has secure_key_bits > 0', rec_a['secure_key_bits'] > 0)
print('\nALL CHECKS PASSED' if checks and all(checks) else '\nSOME CHECKS FAILED — see /tmp/seq_*.log on the nodes')

## Notes

- The parity exchange now rides the real WAN, so a high-latency link makes reconciliation take longer (many round trips) — a real networking cost you can measure with `df.apply_classical_netem`.
- Above ~11% QBER the run aborts instead of reconciling; try `eve_fraction=1.0` in `run_sequence_bb84` to see it.